In [100]:
# import fastf1
import pandas as pd
import os
import sys
import importlib
import src.fantasy

importlib.reload(src.fantasy)

sys.path.append(os.path.abspath(".."))

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from src.fantasy import build_pantasy_team
from src.data_loader import save_dataset, load_dataset

# fastf1.Cache.enable_cache("../data/cache")

In [101]:
df_2023 = load_dataset("../data/processed/race_results_2023.csv")
df_2024 = load_dataset("../data/processed/race_results_2024.csv")

df = pd.concat([df_2023, df_2024], ignore_index=True)

In [102]:
# schedule_2023 = fastf1.get_event_schedule(2023)

# all_races_2023 = []

# for _, event in schedule_2023.iterrows():
#     try:
#         race_name = event["EventName"]

#         session = fastf1.get_session(2023, race_name, "R")
#         session.load()

#         race_results = session.results[[
#             "Abbreviation",
#             "TeamName",
#             "GridPosition",
#             "Position",
#             "Status"
#         ]].copy()

#         race_results["RaceName"] = race_name
#         race_results["RoundNumber"] = event["RoundNumber"]
#         race_results["Year"] = 2023

#         all_races_2023.append(race_results)

#         print(f"Successfully loaded {race_name}")
#     except Exception as e:

#         print(f"Error loading {race_name}: {e}")

# df_2023 = pd.concat(all_races_2023, ignore_index=True)

# save_dataset(
#     df_2023,
#     "../data/processed/race_results_2023.csv"
# )
# # df_2023.to_csv("../data/f1_2023_clean.csv", index=False)
# # print("Data saved into 2023 csv")


In [103]:
# schedule_2024 = fastf1.get_event_schedule(2024)

# all_races_2024 = []

# for _, event in schedule_2024.iterrows():
#     try:
#         race_name = event["EventName"]

#         session = fastf1.get_session(2024, race_name, "R")
#         session.load()

#         race_results = session.results[[
#             "Abbreviation",
#             "TeamName",
#             "GridPosition",
#             "Position",
#             "Status"
#         ]].copy()

#         race_results["RaceName"] = race_name
#         race_results["RoundNumber"] = event["RoundNumber"]
#         race_results["Year"] = 2024

#         all_races_2024.append(race_results)

#         print(f"Successfully loaded {race_name}")
#     except Exception as e:

#         print(f"Error loading {race_name}: {e}")

# df_2024 = pd.concat(all_races_2024, ignore_index=True)

# save_dataset(
#     df_2024,
#     "../data/processed/race_results_2024.csv"
# )

# # df_2024.to_csv("../data/f1_2024_clean.csv", index=False)
# # print("Data saved into 2024 csv")

In [104]:
# to avoid making features manually we can make a function for it

def add_features(df):
    df = df[df["Status"] == "Finished"].copy()

    df = df.sort_values(by=["Abbreviation", "Year", "RoundNumber"])

    df["PreviousPosition"] = (
        df.groupby("Abbreviation")["Position"]
        .shift(1)
    )

    df["Rolling3Average"] = (
        df.groupby("Abbreviation")["Position"]
        .rolling(window=3)
        .mean()
        .reset_index(0, drop=True)
    )

    df = df.dropna(subset=["PreviousPosition", "Rolling3Average"])

    return df

In [105]:
df_2023 = add_features(df_2023)
df_2024 = add_features(df_2024)

In [106]:
df_2023["PositionChange"] = df_2023["GridPosition"] - df_2023["Position"]
df_2024["PositionChange"] = df_2024["GridPosition"] - df_2024["Position"]

df_2023 = df_2023.sort_values(by=["Abbreviation", "RoundNumber"])
df_2024 = df_2024.sort_values(by=["Abbreviation", "RoundNumber"])

In [107]:
df_2023["AveragePositionChange"] = (
    df_2023.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).mean())
)

df_2024["AveragePositionChange"] = (
    df_2024.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).mean())
)

df_2023["AveragePositionChange"] = df_2023["AveragePositionChange"].fillna(0)
df_2024["AveragePositionChange"] = df_2024["AveragePositionChange"].fillna(0)

In [108]:
df_2023["Consistency"] = (
    df_2023.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).std())
)

df_2024["Consistency"] = (
    df_2024.groupby("Abbreviation")["PositionChange"]
    .transform(lambda x: x.rolling(window=3).std())
)

df_2023["Consistency"] = df_2023["Consistency"].fillna(0)
df_2024["Consistency"] = df_2024["Consistency"].fillna(0)


In [109]:
df_2023["GridvsForm"] = df_2023["GridPosition"] - df_2023["Rolling3Average"]
df_2024["GridvsForm"] = df_2024["GridPosition"] - df_2024["Rolling3Average"]
# since rolling 3 average is very dominating
# we can use that feature to make new features
# we can check the grid position against the form of the driver 
# this will let us see if the driver is overperforming or underperforming compared to their recent form

df_2023["FormTrend"] = df_2023["Rolling3Average"] - df_2023["PreviousPosition"]
df_2024["FormTrend"] = df_2024["Rolling3Average"] - df_2024["PreviousPosition"]

features = [
    "TeamName",
    "GridPosition",
    "PreviousPosition",
    "AveragePositionChange",
    "Consistency",
    "GridvsForm",
    "FormTrend",
    "Position"
]

train_2023 = df_2023[features].copy()
test_2024 = df_2024[features].copy()

combined_df = pd.concat([train_2023, test_2024], ignore_index=True)
combined_df = pd.get_dummies(combined_df, columns=["TeamName"])

train_encoded = combined_df.iloc[:len(train_2023)]
test_encoded = combined_df.iloc[len(train_2023):]

X_train = train_encoded.drop("Position", axis=1)
y_train = train_encoded["Position"]

X_test = test_encoded.drop("Position", axis=1)
y_test = test_encoded["Position"]

print("Rolling3Average" in X_train.columns)
print(X_train.columns)



False
Index(['GridPosition', 'PreviousPosition', 'AveragePositionChange',
       'Consistency', 'GridvsForm', 'FormTrend', 'TeamName_Alfa Romeo',
       'TeamName_AlphaTauri', 'TeamName_Alpine', 'TeamName_Aston Martin',
       'TeamName_Ferrari', 'TeamName_Haas F1 Team', 'TeamName_Kick Sauber',
       'TeamName_McLaren', 'TeamName_Mercedes', 'TeamName_RB',
       'TeamName_Red Bull Racing', 'TeamName_Williams'],
      dtype='object')


In [110]:
model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

mean_error = mean_absolute_error(y_test, prediction)
print(f"Mean Absolute Error: {mean_error:.2f}")

# HERE WE REMOVED THE ROLLING 3 AVERAGE FEATURE
# BECAUSE THAT FEATURE BASICALLY DOMINATED THE ENTIRE MODEL
# NOW THE ERROR HAS INCREASED BUT WE CAN SEE OTHER FEATURES HAVE MORE IMPORTANCE NOW

Mean Absolute Error: 1.49


In [111]:
test_info = df_2024[["Abbreviation", "TeamName", "RaceName", "RoundNumber", "Position", "GridPosition", "AveragePositionChange"]].copy()

In [112]:
fantasy_table = test_info.copy()

fantasy_table["Predicted"] = prediction
fantasy_table["PredictedRounded"] = fantasy_table["Predicted"].round().astype(int)
# forgot to explain earlier, here we round up the predicted position
# because value we get is a continuous value and we want to convert it to a discrete position

fantasy_table["Confidence"] = abs(fantasy_table["Predicted"] - fantasy_table["PredictedRounded"])
# THIS SHOWS US HOW CERTAIN OUR PREDICTION IS
# CERTAINTY ABOUT THE DRIVER'S FINISHING POSITION
# WE NEED A VERY LOW SCORE FOR IT TO HAVE VERY HIGH CONFIDENCE
# Something that is as close to 0 as possible means its good, this shows that the model is CONFIDENT that the driver is around this position 

fantasy_table["Error"] = (fantasy_table["Position"] - fantasy_table["Predicted"]).abs()

fantasy_table = fantasy_table.sort_values(by="Predicted")

fantasy_table.head(20)

# this table is what we will use for the fantasy predictor

,Abbreviation,TeamName,RaceName,RoundNumber,Position,GridPosition,AveragePositionChange,Predicted,PredictedRounded,Confidence,Error
79,VER,Red Bull Racing,Japanese Grand Prix,4,1.0,1.0,0.000000,1.00,1,0.00,0.00
99,VER,Red Bull Racing,Chinese Grand Prix,5,1.0,1.0,0.000000,1.00,1,0.00,0.00
139,VER,Red Bull Racing,Emilia Romagna Grand Prix,7,1.0,1.0,-0.333333,1.04,1,0.04,0.04
40,VER,Red Bull Racing,Saudi Arabian Grand Prix,2,1.0,1.0,0.000000,1.05,1,0.05,0.05
120,VER,Red Bull Racing,Miami Grand Prix,6,2.0,1.0,-0.333333,1.07,1,0.07,0.93
80,PER,Red Bull Racing,Japanese Grand Prix,4,2.0,2.0,0.000000,1.53,2,0.47,0.47
199,VER,Red Bull Racing,Spanish Grand Prix,10,1.0,2.0,0.666667,1.56,2,0.44,0.56
101,PER,Red Bull Racing,Chinese Grand Prix,5,3.0,2.0,0.000000,1.58,2,0.42,1.42
381,VER,Red Bull Racing,United States Grand Prix,19,3.0,2.0,0.000000,1.58,2,0.42,1.42
223,VER,Red Bull Racing,Austrian Grand Prix,11,5.0,1.0,-0.666667,1.66,2,0.34,3.34


In [113]:
# Now we try something different
# We try to pick a race and then see what the predictions are

# race_name = "Italian Grand Prix"

# race_predictions = fantasy_table[
#     fantasy_table["RaceName"] == race_name
# ].copy()

# race_predictions.sort_values(by="Position").head(20)

# Here we provide the race name and use the fantasy table to see the predictions
# And then we sort by the position cuz i believe its easier
# And there is pretty good correlation between the predicted and actual positions

In [114]:
safe_picks = race_predictions.sort_values(by="Predicted").head(3)

print(safe_picks[["Abbreviation", "TeamName", "Position", "Predicted", "PredictedRounded"]])

    Abbreviation  TeamName  Position  Predicted  PredictedRounded
424          NOR   McLaren       6.0       3.07                 3
423          LEC   Ferrari       5.0       3.23                 3
422          RUS  Mercedes       4.0       4.29                 4


In [115]:
midfield = race_predictions[
    (race_predictions["Predicted"] > 5) &
    (race_predictions["Predicted"] <= 12)
].copy()


midfield["GridGap"] = midfield["GridPosition"] - midfield["Predicted"]
# New feature that we are adding into v3 and we check if the driver finished better or worse than they started

midfield["GainerScore"] = (
   0.6 *  midfield["AveragePositionChange"]
    - 0.2 * midfield["Predicted"]
    - 0.1 * midfield["Confidence"]
    + 0.4 * midfield["GridGap"]
)
# This gainerscore is vital as it is the decision brain
# AveragePositionChange -> we check the amount of the position gained/lost
# Predicted is the expected finishing position, so lower = better, 
# Confidence controls the risk, high conf = high uncertainty, therefore we penalize the unstable predictions
# THIS FORMULA IS HEURISTIC
# The weights will determine the behavior,
# If predicted penalty increases it picks safer drivers
# If confidence penalty increases it avoids uncertain picks
# PositionChange increase, then picks high upside drivers

midfield = midfield[
    midfield["Confidence"] < 0.5
]

stable_midfield = midfield.sort_values(
    by="GainerScore",
    ascending=False
).head(2)

risky_pool = midfield[
    ~midfield["Abbreviation"].isin(stable_midfield["Abbreviation"])
]
# This takes all the midfield drivers except the ones in stable midfield, isin checks it
# tilde is meant as NOT

risky_pick = risky_pool.sort_values(
    by="AveragePositionChange",
    ascending=False
).head(1)


position_gains_v3 = pd.concat([
    stable_midfield,
    risky_pick
])

position_gains_v3[[
    "Abbreviation",
    "TeamName",
    "GridPosition",
    "GridGap",
    "Position",
    "Predicted",
    "PredictedRounded",
    "AveragePositionChange",
    "Confidence",
    "GainerScore"
]]

,Abbreviation,TeamName,GridPosition,GridGap,Position,Predicted,PredictedRounded,AveragePositionChange,Confidence,GainerScore
419,VER,Red Bull Racing,17.0,11.84,1.0,5.16,5,3.666667,0.16,5.888
421,GAS,Alpine,13.0,6.86,3.0,6.14,6,3.333333,0.14,3.502
426,PIA,McLaren,8.0,-0.30,8.0,8.30,8,3.000000,0.30,-0.010


In [116]:
fantasy_team_v3 = pd.concat([
    safe_picks,
    position_gains_v3
]).drop_duplicates()

fantasy_team_v3[[
    "Abbreviation",
    "TeamName",
    "Position",
    "Predicted",
    "PredictedRounded",
    "AveragePositionChange",
    "Confidence",
    "GainerScore"
]]

,Abbreviation,TeamName,Position,Predicted,PredictedRounded,AveragePositionChange,Confidence,GainerScore
424,NOR,McLaren,6.0,3.07,3,-2.333333,0.07,NaN
423,LEC,Ferrari,5.0,3.23,3,1.666667,0.23,NaN
422,RUS,Mercedes,4.0,4.29,4,4.000000,0.29,NaN
419,VER,Red Bull Racing,1.0,5.16,5,3.666667,0.16,5.888
421,GAS,Alpine,3.0,6.14,6,3.333333,0.14,3.502
426,PIA,McLaren,8.0,8.30,8,3.000000,0.30,-0.010


In [117]:
def evaluate_fantasy_picks(fantasy_team):
    total_picks = len(fantasy_team)

    top5_hits = (fantasy_team["Position"] <= 5).sum()
    top10_hits = (fantasy_team["Position"] <= 10).sum()
    average_actual_finish = fantasy_team["Position"].mean()
    average_error = (abs(fantasy_team["Position"] - fantasy_team["Predicted"])).mean()

    return {
        "Total picks": int(total_picks),
        "Top 5 picks": int(top5_hits),
        "Top 5 hit rate": round(top5_hits / total_picks, 2), 
        "Top 10 picks": int(top10_hits),
        "Top 10 hit rate": round(top10_hits / total_picks, 2),
        "Average Actual Finish": round(average_actual_finish, 2),
        "Average Prediction Error": round(average_error, 2)
    }

In [118]:
all_v3_results = []

for race_name in fantasy_table["RaceName"].unique():
    race_predictions = fantasy_table[
        fantasy_table["RaceName"] == race_name
    ].copy()

    safe_picks = race_predictions.sort_values(by="Predicted").head(3)

    midfield = race_predictions[
        (race_predictions["Predicted"] > 5) &
        (race_predictions["Predicted"] <= 12)
    ].copy()

    midfield["GridGap"] = midfield["GridPosition"] - midfield["Predicted"]

    midfield["GainerScore"] = (
    0.6 *  midfield["AveragePositionChange"]
    - 0.2 * midfield["Predicted"]
    - 0.1 * midfield["Confidence"]
    + 0.4 * midfield["GridGap"]
    )

    midfield = midfield[midfield["Confidence"] < 0.5]

    stable_midfield = midfield.sort_values(
        by="GainerScore",
        ascending=False
    ).head(2)

    risky_pool = midfield[
        ~midfield["Abbreviation"].isin(stable_midfield["Abbreviation"])
    ]

    risky_pick = risky_pool.sort_values(
        by="AveragePositionChange",
        ascending=False
    ).head(1)

    position_gains_v3 = pd.concat([stable_midfield, risky_pick])

    fantasy_team_v3 = pd.concat([
        safe_picks,
        position_gains_v3
    ]).drop_duplicates()

    race_results_v3 = evaluate_fantasy_picks(fantasy_team_v3)

    race_results_v3["RaceName"] = race_name
    race_results_v3["Picks"] = ", ".join(
        fantasy_team_v3["Abbreviation"].tolist()
    )

    all_v3_results.append(race_results_v3)

v3_results_df = pd.DataFrame(all_v3_results)

v3_results_df.to_csv("../results/fantasy_v3_results.csv", index=False)

v3_results_df

,Total picks,Top 5 picks,Top 5 hit rate,Top 10 picks,Top 10 hit rate,Average Actual Finish,Average Prediction Error,RaceName,Picks
0,6,4,0.67,6,1.00,4.17,0.69,Japanese Grand Prix,"VER, PER, NOR, LEC, RUS, ALO"
1,6,3,0.50,5,0.83,6.33,1.86,Chinese Grand Prix,"VER, PER, LEC, HAM, RUS, STR"
2,6,3,0.50,6,1.00,4.50,0.88,Emilia Romagna Grand Prix,"VER, NOR, LEC, PER, HAM, RUS"
3,5,3,0.60,5,1.00,4.60,0.46,Saudi Arabian Grand Prix,"VER, LEC, PIA, RUS, HAM"
4,6,3,0.50,6,1.00,5.33,1.21,Miami Grand Prix,"VER, PER, LEC, ALO, HAM, RUS"
5,6,4,0.67,6,1.00,4.33,0.54,Spanish Grand Prix,"VER, NOR, HAM, PER, PIA, LEC"
6,6,3,0.50,5,0.83,5.33,0.90,United States Grand Prix,"VER, NOR, LEC, RUS, PER, MAG"
7,6,4,0.67,6,1.00,4.00,2.06,Austrian Grand Prix,"VER, RUS, SAI, PIA, HUL, PER"
8,6,4,0.67,6,1.00,4.33,0.98,Dutch Grand Prix,"VER, NOR, PIA, HAM, SAI, PER"
9,6,4,0.67,6,1.00,3.83,1.22,Singapore Grand Prix,"VER, NOR, PIA, SAI, HAM, RUS"


In [119]:
race_results_v3 = evaluate_fantasy_picks(fantasy_team_v3)

race_results_v3["RaceName"] = race_name
race_results_v3["Picks"] = ", ".join(fantasy_team_v3["Abbreviation"].tolist())

race_results_v3

{'Total picks': 6,
 'Top 5 picks': 4,
 'Top 5 hit rate': np.float64(0.67),
 'Top 10 picks': 6,
 'Top 10 hit rate': np.float64(1.0),
 'Average Actual Finish': np.float64(4.5),
 'Average Prediction Error': np.float64(2.1),
 'RaceName': 'São Paulo Grand Prix',
 'Picks': 'NOR, LEC, RUS, VER, GAS, PIA'}

In [125]:
team = build_pantasy_team(
    fantasy_table,
    "Canadian Grand Prix"
)

team_display = team.copy()

team_display["GainerScore"] = team_display["GainerScore"].fillna("Safe Pick")

team_display = team_display.round(2)

team_display

,Abbreviation,TeamName,RaceName,RoundNumber,Position,GridPosition,AveragePositionChange,Predicted,PredictedRounded,Confidence,Error,GridGap,GainerScore
179,VER,Red Bull Racing,Canadian Grand Prix,9,1.0,2.0,0.33,1.89,2,0.11,0.89,0.11,Safe Pick
180,NOR,McLaren,Canadian Grand Prix,9,2.0,3.0,0.33,2.41,2,0.41,0.41,0.59,Safe Pick
181,RUS,Mercedes,Canadian Grand Prix,9,3.0,1.0,-1.00,3.09,3,0.09,0.09,-2.09,Safe Pick
182,HAM,Mercedes,Canadian Grand Prix,9,4.0,7.0,1.67,6.74,7,0.26,2.74,0.26,-0.27
187,GAS,Alpine,Canadian Grand Prix,9,9.0,15.0,0.00,11.37,11,0.37,2.37,3.63,-0.860167
184,ALO,Aston Martin,Canadian Grand Prix,9,6.0,6.0,0.67,7.13,7,0.13,1.13,-1.13,-1.491


In [121]:
# file_path = "../results/fantasy_v3_results.csv"

# race_results_v3_df = pd.DataFrame([race_results_v3])

# if os.path.exists(file_path):
#     existing_df = pd.read_csv(file_path)
#     existing_df = existing_df[existing_df["RaceName"] != race_name]

#     updated_df = pd.concat([existing_df, race_results_v3_df], ignore_index=False)
#     updated_df.to_csv(file_path, index=False)
# else:
#     race_results_v3_df.to_csv(file_path, index=False)

Our main goal for this project is basically finding set of drivers that provide us the best possible upsides.
Upside => the most value that we can get, for example, if a driver starts at P3 and finishes P3, good race for them but fantasy wise it is not TOO impactful.
If driver starts P10 and finishes P4 thats huge upside and BIG value 

**GridGap** now helps us modify the gainer score calculations to make the upside value more effective
This lets us decide which driver will give us the best values. If a race becomes more chaotic, filled with unpredictable moments then this addition of gridgap is helpful.

### Key thing to note

This version of the model will consider the driver with high upside while v2 just picked a stable driver. Here we check for their actual value from when they started to when they finished.